In [ ]:
import os
import random
import time
import pandas as pd

# ==============================================================================
# RIGOROUS HYBRID OPTIMIZER - UNFILTERED (FULL-NETWORK CONTROL EXPERIMENT)
# ==============================================================================

def get_all_network_nodes(rules_text):
    """Extracts ALL unique genes (targets and sources) present in the rules file."""
    all_genes = set()
    BLACKLIST = {"False", "True", "0", "1", "and", "or", "not", "(", ")"}
    
    for line in rules_text.strip().split('\n'):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '*=' in line:
            target, rule = line.split('*=', 1)
            all_genes.add(target.strip())
            
            clean_r = rule.replace("(", " ").replace(")", " ")
            for word in clean_r.split():
                if word not in BLACKLIST and not word.isdigit():
                    all_genes.add(word)
                    
    return sorted(list(all_genes))

def parse_rules(rules_text):
    """Parses network rules and compiles them into executable Python eval() code blocks."""
    rules = {}
    compiled_rules = {}
    gene_list = []
    for line in rules_text.strip().split('\n'):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '*=' in line:
            p = line.split('*=')
            gene, formula = p[0].strip(), p[1].strip()
            rules[gene] = formula
            gene_list.append(gene)
            compiled_rules[gene] = compile(formula, '<string>', 'eval')
    return rules, compiled_rules, gene_list

def simulate_to_attractor(rules, compiled_rules, gene_list, initial_state, perturbations, max_steps=2000):
    """
    RIGOROUS SIMULATION PROFILE:
    - No exceptions or errors are suppressed.
    - If a boolean rule cannot be evaluated, a fatal execution exception is raised.
    """
    curr = initial_state.copy()
    curr.update(perturbations)
    trajectory = []
    
    for step in range(max_steps):
        sig = tuple(curr.get(g, 0) for g in gene_list)
        trajectory.append(sig)
        
        next_state = curr.copy()
        for g, code in compiled_rules.items():
            if g in perturbations:
                # Constrained node: remains forced at the designated state
                continue
            try:
                val = eval(code, {"__builtins__": {}}, curr)
                next_state[g] = 1 if val else 0
            except Exception as e:
                print("\n❌ FATAL ERROR DURING STATE UPDATE")
                print(f"  Step:      {step}")
                print(f"  Gene:      {g}")
                print(f"  Equation:  {rules.get(g, 'UNKNOWN')}")
                print(f"  State key present: {g in curr}")
                print(f"  Python error: {e}")
                raise
        
        curr = next_state
        current_sig = tuple(curr.get(g, 0) for g in gene_list)
        
        if current_sig in trajectory:
            start_idx = trajectory.index(current_sig)
            attractor_sigs = trajectory[start_idx:]
            return [dict(zip(gene_list, s)) for s in attractor_sigs]
    
    # Target attractor state not resolved within max_steps limits
    print(f"\n⚠️ WARNING: No attractor detected within {max_steps} steps.")
    print("   Returning last 50 states as an approximation of late dynamics.")
    tail = trajectory[-50:] if len(trajectory) > 50 else trajectory
    return [dict(zip(gene_list, s)) for s in tail]

def evaluate_phenotype(attractor_states, target_list):
    """Evaluates target nodes stability and compliance criteria across the resolved attractor."""
    score = 0
    target_results = {}
    
    for target, desired_val in target_list.items():
        vals = [state.get(target, -1) for state in attractor_states]
        is_stable_and_correct = all(v == desired_val for v in vals)
        if is_stable_and_correct:
            score += 1
            target_results[target] = "CORRECT"
        else:
            unique_vals = list(set(vals))
            if len(unique_vals) > 1:
                target_results[target] = "❌ OSCILLATES (FAILED)"
            else:
                target_results[target] = "❌ FIXED WRONG"
    return score, target_results

def run_hybrid_optimization_unfiltered(rules_text, target_list, patient_csv, trap_space_constraints,
                                       max_perturbations=8, pop_size=100, generations=50):
    print("="*75)
    print("STARTING STRICT HYBRID OPTIMIZER - UNFILTERED FULL NETWORK")
    print("="*75)
    
    start_time = time.time()
    
    rules, compiled_rules, gene_list_targets = parse_rules(rules_text)
    all_network_genes = get_all_network_nodes(rules_text)
    
    # 1. LOAD PATIENT DATA
    patient_state = {g: 0 for g in all_network_genes}
    loaded = 0
    if os.path.exists(patient_csv):
        df_initial = pd.read_csv(patient_csv)
        if 'GeneSymbol' in df_initial.columns and 'Boolean' in df_initial.columns:
            initial_map = dict(zip(df_initial['GeneSymbol'], df_initial['Boolean']))
            for g in all_network_genes:
                val = initial_map.get(g, None)
                if val is not None:
                    patient_state[g] = 1 if str(val).lower() in ['1', '1.0', 'true'] else 0
                    loaded += 1
            print(f"✅ Patient loaded: {patient_csv} (Mapped {loaded}/{len(all_network_genes)} genes)")
        else:
            print("❌ Error: CSV must contain 'GeneSymbol' and 'Boolean' columns.")
            return
    else:
        print(f"❌ Error: File {patient_csv} not found.")
        return

    # 2. BASELINE EVALUATION
    print("\n--- PHASE 1: BASELINE EVALUATION (Natural Unperturbed Network) ---")
    baseline_attractor = simulate_to_attractor(
        rules, compiled_rules, all_network_genes, patient_state, {}, max_steps=2000
    )
    baseline_score, baseline_results = evaluate_phenotype(baseline_attractor, target_list)
    print(f"Natural Baseline Score: {baseline_score} / {len(target_list)}")
    
    # 3. PREPARE UNFILTERED SEARCH SPACE
    eligible_candidates = [c for c in all_network_genes if c not in target_list and c not in trap_space_constraints]
    print(f" Search Space (UNFILTERED): {len(eligible_candidates)} candidate genes.")

    # 4. ITERATIVE GENETIC ALGORITHM
    print(f"\n--- PHASE 2: STRICT GENETIC SEARCH (Iterative Deepening up to {max_perturbations} nodes) ---")
    
    global_best_score = -1
    global_best_individual = None
    global_best_results = None
    global_best_k = 0
    
    for k in range(1, max_perturbations + 1):
        print(f"\n> Phase {k}: Testing exactly {k} perturbations...")
        
        population = []
        for _ in range(pop_size):
            genes = random.sample(eligible_candidates, min(k, len(eligible_candidates)))
            population.append({g: random.choice([0, 1]) for g in genes})
            
        local_best_score = -1
        local_best_individual = None
        local_best_results = None
        
        for gen in range(generations):
            scored_pop = []
            for ind in population:
                full_perturbation = ind.copy()
                full_perturbation.update(trap_space_constraints)
                
                attr = simulate_to_attractor(
                    rules, compiled_rules, all_network_genes, patient_state, full_perturbation, max_steps=2000
                )
                raw_score, res = evaluate_phenotype(attr, target_list)
                
                caspase_bonus = 0
                for c in ['CASP3', 'CASP7', 'CASP8', 'CASP9']:
                    if res.get(c) == "CORRECT":
                        caspase_bonus += 0.1 
                
                scored_pop.append((raw_score + caspase_bonus, raw_score, ind, res))
                
                if raw_score > local_best_score:
                    local_best_score = raw_score
                    local_best_individual = ind
                    local_best_results = res
                    
            scored_pop.sort(key=lambda x: x[0], reverse=True)
            next_gen = [ind for total_score, raw_score, ind, res in scored_pop[:max(1, pop_size//5)]]
            
            while len(next_gen) < pop_size and scored_pop:
                p1 = random.choice(scored_pop[:max(1, pop_size//2)])[2]
                p2 = random.choice(scored_pop[:max(1, pop_size//2)])[2]
                child = {}
                combined_genes = list(set(list(p1.keys()) + list(p2.keys())))
                if combined_genes:
                    selected_genes = random.sample(combined_genes, min(k, len(combined_genes)))
                    for g in selected_genes:
                        if g in p1 and random.random() > 0.5:
                            child[g] = p1[g]
                        else:
                            child[g] = p2.get(g, random.choice([0, 1]))
                if random.random() < 0.2 and eligible_candidates:
                    mut_gene = random.choice(eligible_candidates)
                    if mut_gene in child:
                        child[mut_gene] = 1 - child[mut_gene]
                    else:
                        if child:
                            k_to_replace = random.choice(list(child.keys()))
                            del child[k_to_replace]
                        child[mut_gene] = random.choice([0, 1])
                next_gen.append(child)
            population = next_gen
            
            if gen % 10 == 0 or gen == generations - 1:
                print(f"  Gen {gen:02d} | Current Best Stable Score: {local_best_score}/{len(target_list)}")
                
            if local_best_score == len(target_list):
                print(f"   Early stop at generation {gen}: perfect raw score reached for k={k}.")
                break 

        if local_best_score > global_best_score:
            global_best_score = local_best_score
            global_best_individual = local_best_individual
            global_best_results = local_best_results
            global_best_k = k
            
        if global_best_score == len(target_list):
            print(f"\n ABSOLUTE STABILITY REACHED! Perfect score obtained with {k} perturbations.")
            break

    end_time = time.time()
    elapsed_time = end_time - start_time

    # 5. FINAL REPORT
    report_lines = []
    report_lines.append("="*75)
    report_lines.append(" UNFILTERED COMPARATIVE REPORT: Baseline VS Optimized Network")
    report_lines.append("="*75)
    
    report_lines.append(f"\n⏱️ COMPUTATIONAL METRICS:")
    report_lines.append(f"  > Search Space Size: {len(eligible_candidates)} genes")
    report_lines.append(f"  > Execution Time:    {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")

    report_lines.append("\nHYBRID THERAPEUTIC STRATEGY (Minimum Guaranteed Nodes):")
    if trap_space_constraints:
        for g, v in trap_space_constraints.items():
            report_lines.append(f"    > {g:<10} -> {v} (Trap Space Constraint)")
    
    if global_best_individual is None:
        report_lines.append("\n⚠️ No valid individual found. Optimization did not improve over baseline.")
        final_report_text = "\n".join(report_lines)
        print("\n" + final_report_text)
        return

    for g, v in global_best_individual.items():
        report_lines.append(f"    > {g:<10} -> {v} (Optimized Target)")
        
    report_lines.append(f"\n OVERALL STABLE IMPROVEMENT:")
    report_lines.append(f"  > Baseline Score:      {baseline_score} / {len(target_list)} targets")
    report_lines.append(f"  > Optimized Score:     {global_best_score} / {len(target_list)} targets")
    report_lines.append(f"  > Total Interventions: {global_best_k} nodes")
    
    report_lines.append("\n TARGET DETAILS (Baseline vs Optimized):")
    report_lines.append(f"{'TARGET':<12} | {'TARGET VAL':<10} | {'BASELINE':<25} | {'OPTIMIZED':<25}")
    report_lines.append("-" * 75)
    for t, desired_val in target_list.items():
        base_res = baseline_results[t]
        opt_res = global_best_results[t]
        
        base_str = "✔️ CORRECT" if base_res == "CORRECT" else base_res
        if opt_res == "CORRECT" and base_res != "CORRECT":
            opt_str = "✅ CORRECT (+)"
        elif opt_res == "CORRECT":
            opt_str = "✔️ CORRECT"
        else:
            opt_str = opt_res
            
        report_lines.append(f"{t:<12} | {desired_val:<10} | {base_str:<25} | {opt_str:<25}")

    final_report_text = "\n".join(report_lines)
    print("\n" + final_report_text)
    
    base_filename = os.path.basename(patient_csv).split('.')[0]
    report_filename = f"UNFILTERED_Report_{base_filename}.txt"
    try:
        with open(report_filename, "w", encoding="utf-8") as file:
            file.write(final_report_text)
        print(f"\n Report saved successfully to: {report_filename}")
    except Exception as e:
        print(f"\n⚠️ Could not save report to file: {e}")

# ==============================================================================
# DATA BLOCK & EXECUTION
# ==============================================================================

if __name__ == "__main__":
    
    RULES_FILE = "rules_pruned_x.txt"
    
    if os.path.exists(RULES_FILE):
        with open(RULES_FILE, "r") as f:
            rules_text = f.read()
    else:
        print(f"❌ Error: Network rules file '{RULES_FILE}' not found!")
        exit()
        
    target_list = {
        'APAF1': 1, 'BAD': 1, 'BCL2': 0, 'BCL2A1': 0, 'BCL2L1': 0,
        'BCL2L11': 1, 'BID': 1, 'BIRC2': 0, 'BIRC5' : 0, 'CASP10': 1,
        'CASP2': 1, 'CASP3': 1, 'CASP6': 1, 'CASP7': 1, 'CASP8': 1,
        'CASP9': 1, 'CYCS' : 1, 'DIABLO': 1, 'FADD': 1, 'MAP3K5': 1,
        'PMAIP1': 1, 'TRADD': 1, 'XIAP': 0, 'BAK1': 1, 'BAX': 1, 'MCL1': 0
    }

    PATIENT_FILE = "TNBC_GSM158xxxx_BOOLEAN_p95.csv"
    
    trap_space_constraints = {}
    
    run_hybrid_optimization_unfiltered(
        rules_text, 
        target_list, 
        PATIENT_FILE, 
        trap_space_constraints, 
        max_perturbations=8,
        pop_size=100,
        generations=50
    )